<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 6 · DATA WAREHOUSING WITH APACHE DORIS</div>

Thirteen new orders arrive, and three of them are wrong: an amount that is not a number, a missing order ID, and a customer who does not exist. You load all thirteen as text with Stream Load, route each to the accepted or the rejected table with SQL rules, trace every rejection back to its raw input, and test the checks by breaking the data on purpose. Module 7 starts from the ten accepted orders. Plan about 30 minutes.

**Module 6: Data Quality and Schema Validation** · [Lesson](course.md) · [Quiz](quiz06_data_quality_and_rejection.ipynb) · [Course README](../../README.md)

## Lab Scope

The Lab rebuilds `orders_raw`, `customers`, the view `orders_classified`, `orders_clean`, and `orders_rejected`. Module 7 reads `orders_clean` and `customers`.

The customer dimension comes from `wwi_customers`, which Lab 5 loads (663 rows). If Lab 5 has not run in this database, the setup cell loads that one table from the committed WWI Parquet package; an existing `wwi_customers` is reused unchanged, and a copy with the wrong row count stops the Lab before anything is reset.

In [ ]:
from pathlib import Path
import sys
import os
from uuid import uuid4

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, expected_failure
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.wwi import manifest, parquet_ddl, parquet_paths
from dw_course.ui import show_sql, show_response

lab = connect_sandbox(module=6)

## 1. Stage the raw input with Stream Load

`datasets/raw_orders.json` holds the thirteen inputs as one JSON array. Each object has an `input_id`, which identifies this arrival, and the order's fields, as the source sent them: `"order_amount": "not-a-number"` in input 11, `"order_id": null` in input 12, `"customer_id": "999999"` in input 13.

`orders_raw` keeps every business column as `VARCHAR`. A text column accepts any text, so Stream Load with `strict_mode` and `max_filter_ratio` = 0, the settings of Lab 5, rejects none of the thirteen rows, and a value that is not a number stays exactly as it arrived. A typed table would reject the bad rows at the door, and their content would be gone. For JSON, Stream Load matches each object's keys to the names in `columns`; `strip_outer_array` says that the file is one array of objects.

The customer dimension `customers` is a Unique Key table copied from `wwi_customers`: an independent list of real customers, not one built from these inputs.

In [ ]:
lab.execute("DROP VIEW IF EXISTS orders_classified")
lab.execute("DROP TABLE IF EXISTS orders_raw")
fields = ",\n    ".join(col + " VARCHAR(100) NULL" for col in ORDER_COLUMNS)
raw_ddl = f'''CREATE TABLE orders_raw (
    input_id BIGINT NOT NULL,
    {fields}
) DUPLICATE KEY(input_id)
DISTRIBUTED BY HASH(input_id) BUCKETS 1
PROPERTIES ("replication_num"="1")'''
show_sql("The raw table: every business column is text", raw_ddl)
lab.execute(raw_ddl)
raw_columns = ",".join(["input_id", *ORDER_COLUMNS])
raw_label = "raw_" + uuid4().hex
show_sql("Stream Load request for the raw input", f"""PUT {os.environ["DW_BE_HTTP_URL"]}/api/{lab.database}/orders_raw/_stream_load
label: {raw_label}
format: json
strip_outer_array: true
columns: {raw_columns}
strict_mode: true
max_filter_ratio: 0
group_commit: off_mode
Request body: datasets/raw_orders.json""")
response = lab.stream_load("orders_raw", COURSE_ROOT / "datasets/raw_orders.json", raw_label, raw_columns, format="json")
show_response(response, title="Raw input Stream Load response")
expect((response["Status"], response["NumberLoadedRows"], response["NumberFilteredRows"]), ("Success", 13, 0),
       title="All thirteen inputs are staged")
lab.sql("SELECT input_id, order_id, customer_id, order_amount, data_source FROM orders_raw WHERE input_id > 9 ORDER BY input_id",
        title="The last four inputs, as they arrived");

lab.execute("DROP TABLE IF EXISTS customers")
lab.execute('CREATE TABLE customers (customer_id BIGINT NOT NULL, customer_name STRING) UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
if not lab.query("SHOW TABLES LIKE 'wwi_customers'"):
    paths = parquet_paths()
    metadata = manifest()["tables"]["customers"]
    lab.execute(parquet_ddl("customers", "wwi_customers"))
    response = lab.stream_load(
        "wwi_customers", paths["customers"], "module6_wwi_" + uuid4().hex, format="parquet"
    )
    show_response(response)
    expect(response["Status"], "Success")
    expect(response["NumberLoadedRows"], metadata["rows"])
    expect(response["NumberFilteredRows"], 0)

lab.execute("INSERT INTO customers SELECT CustomerID, CustomerName FROM wwi_customers")
expect(lab.query("SELECT COUNT(*) FROM customers"), [(663,)], title="The customer dimension has 663 customers")

**Expected result**

The table definition, the request, and its response: `Status` Success, `NumberLoadedRows` 13, `NumberFilteredRows` 0, and a green check. The last four inputs show the problems as text: input 10 is a normal order (900010, customer 10, 250.00), input 11 has the amount `not-a-number`, input 12 has no order ID (`NULL`), and input 13 has the customer `999999`. All four are `COURSE_SIMULATION`.

Then a green check for 663 customers. If Lab 5 had not run in this database, the cell first loads `wwi_customers` and shows that load's response, 663 rows.

## 2. Route Records Using Fixed Rules

The classification view attempts field conversions with TRY_CAST, checks the order ID, amount, customer reference, and source in sequence, and records the first matching rejection reason. Then write accepted rows to orders_clean and problematic input_id values and their reasons to orders_rejected.

Expect ten accepted rows and three rejected rows; join orders_raw on input_id to inspect the original text of each error.

In [ ]:
lab.execute("""
CREATE VIEW orders_classified AS
SELECT *, CASE
    WHEN TRY_CAST(order_id AS BIGINT) IS NULL THEN 'INVALID_ORDER_ID'
    WHEN TRY_CAST(order_amount AS DECIMAL(12,2)) IS NULL
      OR TRY_CAST(order_amount AS DECIMAL(12,2)) < 0 THEN 'INVALID_AMOUNT'
    WHEN TRY_CAST(customer_id AS BIGINT) IS NULL THEN 'INVALID_CUSTOMER'
    WHEN TRY_CAST(customer_id AS BIGINT) NOT IN (SELECT customer_id FROM customers) THEN 'INVALID_CUSTOMER'
    WHEN data_source IS NULL OR data_source <> 'COURSE_SIMULATION' THEN 'INVALID_SOURCE'
    ELSE NULL END AS reject_reason
FROM orders_raw
""")
lab.execute("DROP TABLE IF EXISTS orders_clean")
ddl = order_ddl("orders_clean")
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS orders_rejected")
lab.execute('CREATE TABLE orders_rejected (input_id BIGINT, reason VARCHAR(32)) DUPLICATE KEY(input_id) DISTRIBUTED BY HASH(input_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.execute(f"INSERT INTO orders_clean ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_classified WHERE reject_reason IS NULL")
lab.execute("INSERT INTO orders_rejected SELECT input_id, reject_reason FROM orders_classified WHERE reject_reason IS NOT NULL")
lab.sql("SELECT reject_reason, COUNT(*) AS input_rows FROM orders_classified GROUP BY reject_reason ORDER BY reject_reason", title="Input routing results");
lab.sql("SELECT r.input_id, r.order_id, r.order_amount, r.customer_id, x.reason FROM orders_rejected x JOIN orders_raw r ON x.input_id=r.input_id ORDER BY r.input_id", title="Three rejected records and their raw fields");

**Expected result**

The definition of `orders_clean`, then the routing result: 10 inputs with no reason (`NULL`), and one each for `INVALID_AMOUNT`, `INVALID_CUSTOMER`, and `INVALID_ORDER_ID`. The three rejected records join back to their raw text: input 11 with `not-a-number`, input 12 with no order ID, and input 13 with customer `999999`.

`999999` converts to an integer without trouble: its type is fine. It is rejected because no customer 999999 exists in `customers`, a business rule that the column's type cannot express.

## 3. Check Business Quality and Inject Errors

First check that all thirteen inputs have a destination, then verify order uniqueness, customer references, amounts, statuses, and complete records. Check event times against the fixed cutoff for this sample batch.

Next, try two counterexamples, calling the uniqueness check directly each time rather than waiting for aggregate checks to fail:

1. Append a duplicate order, then query and display the duplicate order ID.
2. After restoring the data, change only the second row's order ID to the first row's order ID, leaving all other fields unchanged. There are still ten rows totaling 1400.00, but one order ID is duplicated and another order is missing.

The uniqueness check must detect both errors. After each experiment, restore the accepted table from the preserved classified input and recheck all fields before handing it over to Module 7.

In [ ]:
expected_rows = order_rows(fixture("orders.json"))
def check_unique_orders():
    expect(lab.query("SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM orders_clean"), [(0,)])

def restore_clean_orders():
    lab.execute("TRUNCATE TABLE orders_clean")
    lab.execute(f"INSERT INTO orders_clean ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_classified WHERE reject_reason IS NULL")

def quality_report():
    expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_clean"), [(10,"1400.00")])
    check_unique_orders()
    expect(lab.query("SELECT COUNT(*) FROM orders_clean WHERE event_time > '2026-01-02 12:00:00' OR event_time IS NULL"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean WHERE status <> 'CREATED' OR paid_amount <> 0 OR refund_amount <> 0"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean o LEFT JOIN customers c ON o.customer_id=c.customer_id WHERE c.customer_id IS NULL"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean WHERE data_source <> 'COURSE_SIMULATION'"), [(0,)])
    # DATE_FORMAT makes the transport representation explicit for fixture comparison.
    projection = ",".join("DATE_FORMAT(event_time, '%Y-%m-%d %H:%i:%s')" if col == "event_time" else col for col in ORDER_COLUMNS)
    expect(lab.query(f"SELECT {projection} FROM orders_clean ORDER BY order_id"), expected_rows,
           title="Quality report: ten unique orders, 1400.00, new, known customers, every field as expected")

expect(lab.query("SELECT input_id, reason FROM orders_rejected ORDER BY input_id"),
       [(11,"INVALID_AMOUNT"),(12,"INVALID_ORDER_ID"),(13,"INVALID_CUSTOMER")], title="Three rejections with their reasons")
expect(lab.query("SELECT COUNT(*) FROM orders_classified WHERE reject_reason IS NULL"), [(10,)], title="Ten inputs pass every rule")
expect(lab.query("SELECT COUNT(*) FROM orders_raw r LEFT JOIN orders_rejected x ON r.input_id=x.input_id WHERE x.input_id IS NULL"), [(10,)],
       title="Every input is either accepted or rejected")
quality_report()

# Trigger the uniqueness check separately so aggregate checks do not interrupt it first.
lab.insert("orders_clean", ORDER_COLUMNS, [expected_rows[0]])
lab.sql("SELECT order_id, COUNT(*) AS copies FROM orders_clean GROUP BY order_id HAVING COUNT(*) > 1 ORDER BY order_id", title="Duplicate order ID after appending");
with expected_failure("Duplicate order check", "Duplicate order detected; the uniqueness rule works"):
    check_unique_orders()
restore_clean_orders()
quality_report()

# A subtler error: the second row uses the first row's order ID, leaving the row count and amount unchanged.
probe_rows = list(expected_rows)
probe_rows[1] = (probe_rows[0][0], *probe_rows[1][1:])
lab.execute("TRUNCATE TABLE orders_clean")
lab.insert("orders_clean", ORDER_COLUMNS, probe_rows)
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_clean"), [(10,"1400.00")], title="The totals still look right: ten rows, 1400.00")
lab.sql("SELECT order_id, COUNT(*) AS copies FROM orders_clean GROUP BY order_id HAVING COUNT(*) > 1 ORDER BY order_id", title="Correct totals, but a duplicate order ID");
with expected_failure("Duplicate check with unchanged totals", "The row count and amount are correct, but the uniqueness check still detects the error"):
    check_unique_orders()
restore_clean_orders()
quality_report()
lab.sql("SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_clean", title="Accepted orders restored after both error experiments");

**Expected result**

Three green checks for the routing, then one card for the quality report, which checks ten accepted orders and 1400.00, no duplicate order ID, every event before the cutoff `2026-01-02 12:00:00`, all CREATED with nothing paid or refunded, every customer present, every order from the simulation, and every field equal to the expected orders.

Then the two experiments. After the appended copy, order 900001 has 2 copies and the uniqueness check fails, shown as a green card: the failure was expected. The data is restored, and the report passes again. In the second experiment, ten rows still total 1400.00, so the totals look right, but 900001 has 2 copies, 900002 is missing, and the uniqueness check fails again. After the second restore, the last table shows 10 orders and 1400.00: the state that Module 7 reads.

## 4. Practice: try a new rule before you use it

The business adds a rule: an accepted order above 200 needs a manual review; the others are accepted. Write a `SELECT` with `CASE` that groups the accepted orders into `ACCEPT` and `REVIEW` with their counts and amounts. Expect 8 orders and 850.00 to accept and 2 orders and 550.00 to review. Only try the rule here: leave `orders_clean` unchanged for Module 7.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
query = """SELECT CASE WHEN order_amount > 200 THEN 'REVIEW' ELSE 'ACCEPT' END AS decision,
COUNT(*) AS orders, SUM(order_amount) AS amount
FROM orders_clean GROUP BY decision ORDER BY decision"""
lab.sql(query, title="New amount review rule");
expect(lab.query(query), [("ACCEPT",8,"850.00"),("REVIEW",2,"550.00")], title="Eight to accept, two to review")
# Only try out the new rule; preserve the ten accepted orders used by Module 7.
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_clean"), [(10,"1400.00")], title="orders_clean is unchanged")

**Expected result**

Two rows: `ACCEPT` with 8 orders and 850.00, `REVIEW` with 2 orders and 550.00, and two green checks, the second showing that `orders_clean` still holds ten orders and 1400.00. A trial `SELECT` changes nothing; to use the rule, you would add it to the classification view and route again.

## You are done when

- all thirteen inputs are staged in `orders_raw`, ten are in `orders_clean` (1400.00), and three are in `orders_rejected`, each traceable to its raw text by `input_id`;
- you can explain why customer `999999` converts to an integer and is still rejected;
- you have seen the uniqueness check fail on two known errors, one of them with correct totals, and pass after each restore.

## Next

Open [Quiz 6: Data Quality and Schema Validation](quiz06_data_quality_and_rejection.ipynb), then continue to [Module 7: Data Updates, Deletion, and Event Replay](../module07-state-changes/course.md).
